# 🗺️ Notebook 03 — Clustering & Hotspot Identification
## ParkSense AI | Gridlock Round 2

**Input:** `data/processed/featured_violations.parquet`  
**Output:** `data/processed/hotspots.parquet` & `clustered_violations.parquet`

This notebook uses DBSCAN to group thousands of individual parking violations into actionable physical 'Hotspots', using strict noise filtering and Geospatial Medoids.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('✅ Imports done.')

✅ Imports done.


In [5]:
# Load the featured dataset
DATA_PATH = '../data/processed/featured_violations.parquet'
df = pd.read_parquet(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows')

coords = df[['latitude', 'longitude']].values
coords_radians = np.radians(coords)
EARTH_RADIUS_KM = 6371.0088

# Strict Hotspot Threshold: 50 violations minimum (approx 2 per week over 6 months)
MIN_SAMPLES = 50
EPSILON_METERS = 50
eps_radians = (EPSILON_METERS / 1000.0) / EARTH_RADIUS_KM

print(f'Running DBSCAN... (eps={EPSILON_METERS}m, min_samples={MIN_SAMPLES})')
dbscan = DBSCAN(eps=eps_radians, min_samples=MIN_SAMPLES, metric='haversine', algorithm='ball_tree', n_jobs=-1)
df['cluster_id'] = dbscan.fit_predict(coords_radians)

n_clusters = df['cluster_id'].nunique() - (1 if -1 in df['cluster_id'].values else 0)
print(f'✅ Found {n_clusters} authentic Hotspots.')

Loaded: 115,353 rows
Running DBSCAN... (eps=50m, min_samples=50)
✅ Found 211 authentic Hotspots.


In [6]:
df_hotspots_raw = df[df['cluster_id'] != -1]

def get_mode(series):
    if len(series) == 0: return None
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else None

# Aggregate features
hotspots = df_hotspots_raw.groupby('cluster_id').agg(
    total_violations=('id', 'count'),
    mean_lat=('latitude', 'mean'),
    mean_lng=('longitude', 'mean'),
    total_pici=('pici_score', 'sum'),
    avg_pici=('pici_score', 'mean'),
    max_pici=('pici_score', 'max'),
    primary_police_station=('police_station', get_mode),
    primary_vehicle_type=('final_vehicle_type', get_mode)
).reset_index()

# Geospatial Medoids: Snap the center to the actual street
center_lats = []
center_lngs = []

print('Calculating Medoids...')
for idx, row in hotspots.iterrows():
    c_id = row['cluster_id']
    mean_lat = row['mean_lat']
    mean_lng = row['mean_lng']
    cluster_points = df_hotspots_raw[df_hotspots_raw['cluster_id'] == c_id][['latitude', 'longitude']].values
    
    if len(cluster_points) > 0:
        nn = NearestNeighbors(n_neighbors=1, metric='haversine')
        nn.fit(np.radians(cluster_points))
        dist, indices = nn.kneighbors(np.radians([[mean_lat, mean_lng]]))
        medoid_pt = cluster_points[indices[0][0]]
        center_lats.append(medoid_pt[0])
        center_lngs.append(medoid_pt[1])
    else:
        center_lats.append(mean_lat)
        center_lngs.append(mean_lng)

hotspots['center_lat'] = center_lats
hotspots['center_lng'] = center_lngs

hotspots = hotspots.sort_values('total_pici', ascending=False).reset_index(drop=True)
hotspots['hotspot_rank'] = hotspots.index + 1

rank_map = hotspots.set_index('cluster_id')['hotspot_rank'].to_dict()
df['hotspot_rank'] = df['cluster_id'].map(rank_map).fillna(-1).astype(int)

HOTSPOTS_OUT_PATH = '../data/processed/hotspots.parquet'
CLUSTERED_OUT_PATH = '../data/processed/clustered_violations.parquet'

hotspots.to_parquet(HOTSPOTS_OUT_PATH, index=False)
df.to_parquet(CLUSTERED_OUT_PATH, index=False)
print('✅ Hotspots summary and clustered violations saved.')

Calculating Medoids...
✅ Hotspots summary and clustered violations saved.
